# Chronos 2 Inference Pipeline
Use this file to test out different feature sets with Chronos-2.
Official documentation: https://github.com/amazon-science/chronos-forecasting

Run the following:
```
pip install 'chronos-forecasting>=2.0' 'pandas[pyarrow]' 'matplotlib' 'scikit-learn' 'numpy'
```


In [ ]:
# Core
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Chronos - 2
from chronos import Chronos2Pipeline

In [ ]:
# ============================
# User Config (EDIT ME)
# ============================

# CSV Loading
csv_path = "YOUR CSV PATH"
date_col = "NAME OF YOUR DATETIME COLUMN"
id_col = None #Set to column name if you have multiple series
default_id_value = "series_1" # Use this if id_col = None

# Target
target_col = "NAME OF YOUR TARGET COLUMN"

# Features
feature_cols = [] # List your feature columns here

# Business Days reindexing
freq = "B"
reindex_start = None
reindex_end = None

# Train / Test split
split_start_date = "2012-01-09" # Train starts here
train_fraction = 0.80 # first 80% context, last 20% forecast horizon

# Chronos-2 model settings
device = "cpu" #if you can use a GPU, enter "cuda"


In [ ]:
# ============================
# Load CSV and Basic Cleaing
# ============================

# load csv
df = pd.read_csv(csv_path)

# ensure datetime column is parsed correctly
df[date_col] = pd.to_datetime(df[date_col])

# Sort by time
df = df.sort_values(date_col).reset_index(drop=True)

# If there is no id column, create a single-id series
if id_col is None:
    df["id"] = default_id_value
    id_col = "id"


# Keep only needed columns
keep_cols = [id_col, date_col, target_col] + feature_cols
df = df[keep_cols].copy()

df.head()

In [ ]:
# ==========================================
# BUSINESS-DAY INDEX + REINDEXING (REQUIRED)
# (done per series to keep Chronos freq valid)
# ==========================================

value_cols = [target_col] + feature_cols

pieces = []

for sid, g in df.groupby(id_col):
    # Sort and remove duplicate timestamps (needed for stable frequency)
    g = g.sort_values(date_col).drop_duplicates(subset=[date_col])

    # Set timestamp index
    g = g.set_index(date_col)

    # Determine business-day range per series
    start_date = reindex_start if reindex_start is not None else g.index.min()
    end_date   = reindex_end   if reindex_end   is not None else g.index.max()
    bdays_idx = pd.date_range(start=start_date, end=end_date, freq=freq)

    # Reindex to business days -> may create NaNs
    g = g.reindex(bdays_idx)

    # Fill ONLY the NaNs created by reindexing
    # This keeps a fully regular business-day calendar.
    # Note!: Currently uses a simple Forward Fill - has to be changed as soon as a better solution is found.
    g[value_cols] = g[value_cols].ffill()

    # Restore id + timestamp column
    g[id_col] = sid
    g.index.name = date_col
    pieces.append(g.reset_index())

# Combine all series back together
df = pd.concat(pieces, ignore_index=True)

print(df.head(10))
print(df.tail(10))


In [ ]:
# Optional Plotting
plt.figure(figsize=(12, 6))
plt.plot(df.index, df[target_col]) # Use df.index instead of df['Time']
plt.xlabel('Time')
plt.ylabel(target_col)
plt.show()

In [ ]:
# ==========================================
# Define context_df + future_df
# ==========================================

# Make sure data is perfectly ordered before splitting
df = df.sort_values([id_col, date_col]).reset_index(drop=True)

# Filter dataset from split_start_date onwards
df_split = df[df[date_col] >= pd.Timestamp(split_start_date)]
df_split = df_split.sort_values(date_col).reset_index(drop=True)

# Compute split points
n_total = len(df_split)
n_train = int(np.floor(train_fraction * n_total))

context_df = df_split.iloc[:n_train].copy()
future_df = df_split.iloc[n_train:].copy()

# Show sizes
print("Context length:", len(context_df))
print("Future length :", len(future_df))
print("----- Context DF ----- ")
print(context_df.head())
print(context_df.tail())
print("----- Future DF ----- ")
print(future_df.head())
print(future_df.tail())

In [8]:
# ===========================
# Load Chronos-2 Model
# ===========================

pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map = device,
)

In [ ]:
# ======================================
# Build  Chronos and inference
# ======================================

"""
Chronos-2 can take two seperate dfs as inputs to perform forecasting. 
The context_df and the future_df. Only the context_df is mandatory.

The context_df includes:
- historical portion of data - including target, features, id, timestamp - that chronos uses to understand past behaviour and patterns.
- must include target
- can include past known covariates and features

The future_df includes:
- only timestamps and optional future-known covaraites for which predictions should be generated
- does NOT inlude the target
- typically represents the forecast horizon --> the part after the split
"""

# prediction length
pred_len = len(future_df)
if pred_len <= 0:
    raise ValueError("future_df is empty, no prediction possible.")

# Build columns sets for Chronos
# Note: in context_cols you have everything needed. in future_cols you just leave out the target_col
context_cols = [id_col, date_col, target_col] + feature_cols
future_cols = [id_col, date_col] + feature_cols

context_df_chronos = context_df[context_cols].reset_index(drop=True)
future_df_chronos = future_df[future_cols].reset_index(drop=True)

print(context_df_chronos.head())
print(future_df_chronos.head())

# Run Chronos inference
pred_df = pipeline.predict_df(
    context_df_chronos,
    future_df = future_df_chronos,
    prediction_length= pred_len,
    quantile_levels = [0.1, 0.5, 0.9],
    id_column = id_col,
    timestamp_column = date_col,
    target = target_col
)

pred_df.head()


In [ ]:
# ======================================
# Merge predictions with the true values
# ======================================

results = pred_df.merge(
    future_df[[id_col, date_col, target_col]],
    on=[id_col, date_col],
    how="left"
)

y_true = results[target_col].astype(float)
y_pred_median = results["0.5"].astype(float)

results.head()


In [ ]:
# ===========================
# Plot
# ===========================

# How many steps were predicted
pred_len = len(results)

# Take exactly the last pred_len historical points
hist_part = context_df[[date_col, target_col]].copy()
hist_part = hist_part.sort_values(date_col).tail(pred_len)

# True future (for the forecast horizon)
true_future_part = future_df[[date_col, target_col]].copy()
true_future_part = true_future_part.sort_values(date_col).head(pred_len)

# Forecast (median + optional intervals)
forecast_part = results[[date_col, "0.5"]].copy().sort_values(date_col)

plt.figure(figsize=(12, 6))

# Plot historical true (same length as forecast)
plt.plot(
    hist_part[date_col],
    hist_part[target_col].astype(float),
    label=f"True {target_col} (last {pred_len} history points)",
    linewidth=1.5
)

# Plot true future values
plt.plot(
    true_future_part[date_col],
    true_future_part[target_col].astype(float),
    label=f"True {target_col} (forecast horizon)",
    linewidth=1.5,
    marker="o"
)

# Plot median forecast
plt.plot(
    forecast_part[date_col],
    forecast_part["0.5"].astype(float),
    label="Chronos median forecast (q0.5)",
    linestyle="--",
    linewidth=2,
    marker="x"
)

# Optional uncertainty band if available
if "0.1" in results.columns and "0.9" in results.columns:
    plt.fill_between(
        results[date_col],
        results["0.1"].astype(float),
        results["0.9"].astype(float),
        alpha=0.2,
        label="10%-90% prediction interval"
    )

# Vertical line at forecast start
split_ts = true_future_part[date_col].min()
plt.axvline(split_ts, linestyle="--", linewidth=1, label="Forecast start")

plt.title("Chronos-2 Forecast vs True Values (Equal History & Forecast Length)")
plt.xlabel("Time")
plt.ylabel(target_col)
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# ===========================
# Metrics
# ===========================

# RMSE
rmse = np.sqrt(mean_squared_error(y_true, y_pred_median))

# MAE
mae = mean_absolute_error(y_true, y_pred_median)

# Adjusted R Squared
r2 = r2_score(y_true, y_pred_median)
n = len(y_true)
p = len(feature_cols)

if n - p - 1 > 0:
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
else:
    adj_r2 = np.nan  # not defined if too few samples vs features

# Quantile Loss (Pinball loss)
valid_mask = ~y_true.isna()
def pinball_loss(y, yhat, q):
    diff = y - yhat
    return np.mean(np.maximum(q * diff, (q - 1)* diff))

quantile_losses = {}
for q in [0.1, 0.5, 0.9]:
    q_str = str(q)
    yhat_q = results[q_str].astype(float)[valid_mask]
    quantile_losses[q] = pinball_loss(y_true.values, yhat_q.values, q)

# CRPS approximation
qs = np.array(sorted(quantile_losses.keys()))
Ls = np.array([quantile_losses[q] for q in qs])
crps = 2 * np.trapezoid(Ls, qs)

# Print
print("CRPS (approx.):", crps)
print("RMSE:", rmse)
print("MAE:", mae)
print("Adjusted R^2:", adj_r2)
print("Quantile Losses:")
for q, Lq in quantile_losses.items():
    print(f"--> q={q}: {Lq}")
